# 00 — Exploratory Data Analysis: French IDS/ADS Vowel Metadata

This notebook explores the metadata CSV produced by `scripts/build_french_vowel_metadata.py`.  
Acoustic columns (`mean_pitch`, `mean_F1`, …) are all `NaN` at this stage; they will be populated once Praat feature extraction is enabled.

**Data:** `outputs/french_vowels_metadata.csv`  
**What this notebook covers:**

1. Dataset overview — shape, columns, missing values  
2. Data quality — unexpected vowel labels, label anomalies  
3. Speaker × session matrix — who has data in which age session  
4. Register distribution — IDS vs ADS by session, activity, and speaker  
5. Vowel inventory — token counts, by register, by session  
6. Activity breakdown — bath / meal / play across register and session  
7. Tokens per speaker — overall count distribution and IDS/ADS balance  
8. Duration distribution — histogram, by vowel, by register

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Style ──────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

REGISTER_PALETTE = {"IDS": "#2196F3", "ADS": "#FF5722"}   # blue / orange
SESSION_ORDER    = ["4m", "8m", "12m"]
ACTIVITY_ORDER   = ["bath", "meal", "play"]

# ── Load data ──────────────────────────────────────────────────────────────
CSV = Path("../outputs/french_vowels_metadata.csv")
df  = pd.read_csv(CSV)

# Enforce categorical ordering
df["session"]  = pd.Categorical(df["session"],  categories=SESSION_ORDER,  ordered=True)
df["activity"] = pd.Categorical(df["activity"], categories=ACTIVITY_ORDER, ordered=True)

print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
print(f"Speakers : {df['speakerid'].nunique()}  |  Sessions : {sorted(df['session'].unique())}  |  Registers : {df['register'].unique().tolist()}")

## 1. Dataset Overview

In [ ]:
print(f"Shape : {df.shape[0]:,} rows  ×  {df.shape[1]} columns\n")
print("Columns:")
for col in df.columns:
    print(f"  {col:<20}  {df[col].dtype}")

print("\nAcoustic columns (expected all-NaN until extraction is enabled):")
acoustic_cols = [c for c in df.columns if c not in [
    "speakerid","session","activity","time","word","vowel","register",
    "start_sec","duration_sec","duration_ms"]]
print(f"  {acoustic_cols}")
print(f"  All NaN: {df[acoustic_cols].isna().all(axis=None)}")

print("\nFirst 5 rows:")
df.head()

## 2. Data Quality — Missing Values and Anomalous Labels

Check for unexpected `NaN` values in the metadata columns and flag vowel labels that look like annotation artefacts (hyphens, extra tokens).

In [ ]:
META_COLS = ["speakerid","session","activity","time","word","vowel","register",
             "start_sec","duration_sec","duration_ms"]

# Missing values in metadata columns
missing = df[META_COLS].isna().sum()
missing = missing[missing > 0]
if missing.empty:
    print("No missing values in metadata columns.")
else:
    print("Missing values found:")
    print(missing)

# Vowel labels that contain a hyphen — likely annotation artefacts
CANONICAL_VOWELS = {"a","e","i","o","u","ai","an","au","en","eu","in","oe","on","ou"}
anomalous = df[~df["vowel"].isin(CANONICAL_VOWELS)][["speakerid","session","word","vowel","register"]]
print(f"\nAnomalous vowel labels ({len(anomalous)} rows):")
print(anomalous.to_string(index=False) if len(anomalous) else "  None found.")

## 3. Speaker × Session Matrix

Which speakers contributed data in each age session?  
A filled cell means the speaker has at least one token; the number shows the token count.

In [ ]:
pivot = (df.groupby(["speakerid","session"], observed=True)
           .size()
           .unstack("session", fill_value=0)
           .reindex(columns=SESSION_ORDER, fill_value=0))

fig, ax = plt.subplots(figsize=(5, 9))
sns.heatmap(pivot, annot=True, fmt="d", cmap="Blues", linewidths=0.4,
            linecolor="white", cbar=False, ax=ax)
ax.set_title("Token count per speaker × session", fontsize=12, pad=10)
ax.set_xlabel("Session (infant age)")
ax.set_ylabel("Speaker ID")
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

# Summary
sessions_per_speaker = (pivot > 0).sum(axis=1)
print(f"\nSpeakers with all 3 sessions : {(sessions_per_speaker == 3).sum()}")
print(f"Speakers with 2 sessions     : {(sessions_per_speaker == 2).sum()}")
print(f"Speakers with 1 session      : {(sessions_per_speaker == 1).sum()}")

## 4. Register Distribution

IDS vs ADS overall, broken down by session and activity.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ── 4a: Overall ────────────────────────────────────────────────────────────
reg_counts = df["register"].value_counts().reindex(["IDS","ADS"])
axes[0].bar(reg_counts.index, reg_counts.values,
            color=[REGISTER_PALETTE[r] for r in reg_counts.index], width=0.5)
axes[0].set_title("Overall")
axes[0].set_ylabel("Token count")
for bar, val in zip(axes[0].patches, reg_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f"{val:,}", ha="center", va="bottom", fontsize=9)

# ── 4b: By session ─────────────────────────────────────────────────────────
reg_by_sess = (df.groupby(["session","register"], observed=True)
                 .size().unstack("register", fill_value=0)
                 .reindex(SESSION_ORDER))
reg_by_sess.plot(kind="bar", ax=axes[1], color=[REGISTER_PALETTE["IDS"],
                 REGISTER_PALETTE["ADS"]], rot=0, width=0.6)
axes[1].set_title("By session")
axes[1].set_xlabel("Session")
axes[1].legend(title="Register", fontsize=8)
axes[1].set_ylabel("")

# ── 4c: By activity ────────────────────────────────────────────────────────
reg_by_act = (df.groupby(["activity","register"], observed=True)
                .size().unstack("register", fill_value=0)
                .reindex(ACTIVITY_ORDER))
reg_by_act.plot(kind="bar", ax=axes[2], color=[REGISTER_PALETTE["IDS"],
                REGISTER_PALETTE["ADS"]], rot=0, width=0.6)
axes[2].set_title("By activity")
axes[2].set_xlabel("Activity")
axes[2].legend(title="Register", fontsize=8)
axes[2].set_ylabel("")

for ax in axes:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

fig.suptitle("IDS / ADS token counts", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# IDS/ADS balance per speaker (stacked bar, sorted by total tokens)
spk_reg = (df.groupby(["speakerid","register"])
             .size().unstack("register", fill_value=0)
             .assign(total=lambda d: d.sum(axis=1))
             .sort_values("total", ascending=True)
             .drop(columns="total"))

fig, ax = plt.subplots(figsize=(5, 10))
spk_reg.plot(kind="barh", stacked=True, ax=ax,
             color=[REGISTER_PALETTE["IDS"], REGISTER_PALETTE["ADS"]],
             width=0.8)
ax.set_title("IDS / ADS tokens per speaker", fontsize=12)
ax.set_xlabel("Token count")
ax.set_ylabel("Speaker ID")
ax.legend(title="Register", loc="lower right", fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## 5. Vowel Inventory

Token counts for each vowel category, split by register and session.

In [ ]:
# Vowel order: descending total token count
vowel_order = df["vowel"].value_counts().index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 5a: Total count per vowel ───────────────────────────────────────────────
vcounts = df["vowel"].value_counts().reindex(vowel_order)
axes[0].bar(vcounts.index, vcounts.values, color="#546E7A", width=0.7)
axes[0].set_title("Total tokens per vowel")
axes[0].set_xlabel("Vowel")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

# ── 5b: Vowel × register (grouped bar) ──────────────────────────────────────
vr = (df.groupby(["vowel","register"])
        .size().unstack("register", fill_value=0)
        .reindex(vowel_order))
vr.plot(kind="bar", ax=axes[1], color=[REGISTER_PALETTE["IDS"],
        REGISTER_PALETTE["ADS"]], rot=45, width=0.7)
axes[1].set_title("Tokens per vowel × register")
axes[1].set_xlabel("Vowel")
axes[1].set_ylabel("")
axes[1].legend(title="Register", fontsize=9)

for ax in axes:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.show()

In [ ]:
# Vowel × session heatmap (proportional within each session so session size
# differences don't dominate the colour scale)
vs = (df.groupby(["session","vowel"], observed=True)
        .size().unstack("vowel", fill_value=0)
        .reindex(SESSION_ORDER)
        .reindex(columns=vowel_order, fill_value=0))

vs_pct = vs.div(vs.sum(axis=1), axis=0) * 100   # row-normalised %

fig, ax = plt.subplots(figsize=(13, 3))
sns.heatmap(vs_pct, annot=vs.values, fmt="d", cmap="YlOrRd",
            linewidths=0.3, linecolor="white", cbar_kws={"label":"%"},
            ax=ax)
ax.set_title("Vowel token counts by session  (cell colour = % within session)", fontsize=11)
ax.set_xlabel("Vowel")
ax.set_ylabel("Session")
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

## 6. Activity Breakdown

Token counts across bath / meal / play by register and session.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── 6a: Activity × register ─────────────────────────────────────────────────
act_reg = (df.groupby(["activity","register"], observed=True)
             .size().unstack("register", fill_value=0)
             .reindex(ACTIVITY_ORDER))
act_reg.plot(kind="bar", ax=axes[0],
             color=[REGISTER_PALETTE["IDS"], REGISTER_PALETTE["ADS"]],
             rot=0, width=0.6)
axes[0].set_title("Tokens per activity × register")
axes[0].set_xlabel("Activity")
axes[0].set_ylabel("Token count")
axes[0].legend(title="Register", fontsize=9)

# ── 6b: Activity × session ──────────────────────────────────────────────────
act_sess = (df.groupby(["activity","session"], observed=True)
              .size().unstack("session", fill_value=0)
              .reindex(ACTIVITY_ORDER)
              .reindex(columns=SESSION_ORDER, fill_value=0))
act_sess.plot(kind="bar", ax=axes[1], colormap="viridis", rot=0, width=0.6)
axes[1].set_title("Tokens per activity × session")
axes[1].set_xlabel("Activity")
axes[1].set_ylabel("")
axes[1].legend(title="Session", fontsize=9)

for ax in axes:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.show()

## 7. Tokens per Speaker

How many tokens does each speaker contribute?  
Also shows IDS/ADS ratio per speaker to spot speakers with unusually low ADS counts.

In [ ]:
spk_total = df.groupby("speakerid").size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── 7a: Distribution of per-speaker token counts ────────────────────────────
axes[0].hist(spk_total.values, bins=15, color="#607D8B", edgecolor="white")
axes[0].axvline(spk_total.median(), color="red", linestyle="--",
                label=f"Median = {spk_total.median():.0f}")
axes[0].set_title("Distribution of token counts per speaker")
axes[0].set_xlabel("Tokens")
axes[0].set_ylabel("Number of speakers")
axes[0].legend(fontsize=9)

# ── 7b: IDS % per speaker ──────────────────────────────────────────────────
ids_pct = (df[df["register"]=="IDS"].groupby("speakerid").size()
           / df.groupby("speakerid").size() * 100).sort_values()

colors = ["#FF5722" if v < 70 else "#2196F3" for v in ids_pct.values]
axes[1].barh(range(len(ids_pct)), ids_pct.values, color=colors)
axes[1].set_yticks(range(len(ids_pct)))
axes[1].set_yticklabels(ids_pct.index, fontsize=8)
axes[1].axvline(ids_pct.median(), color="black", linestyle="--",
                label=f"Median = {ids_pct.median():.1f}%")
axes[1].set_title("IDS % per speaker  (red = < 70%)")
axes[1].set_xlabel("% IDS tokens")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nPer-speaker token count — min:{spk_total.min()}  median:{spk_total.median():.0f}  max:{spk_total.max()}")
print(f"Speakers with IDS% < 70% : {(ids_pct < 70).sum()}")

## 8. Vowel Duration Distribution

Duration is the only numeric measure available at this stage.  
We look at the overall distribution, outliers, and differences across vowels and registers.

In [ ]:
# Cap display at 500 ms so long-tail outliers don't compress the main body
DUR_CAP = 500
df_plot = df[df["duration_ms"] <= DUR_CAP]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── 8a: Overall histogram ──────────────────────────────────────────────────
axes[0].hist(df_plot["duration_ms"], bins=60, color="#546E7A", edgecolor="white")
for p, label in [(df["duration_ms"].median(), "median"),
                 (df["duration_ms"].quantile(0.75), "Q3"),
                 (df["duration_ms"].quantile(0.95), "P95")]:
    axes[0].axvline(p, linestyle="--", label=f"{label} = {p:.0f} ms")
axes[0].set_title(f"Duration distribution (capped at {DUR_CAP} ms)")
axes[0].set_xlabel("Duration (ms)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=9)

# ── 8b: IDS vs ADS ────────────────────────────────────────────────────────
for reg, col in REGISTER_PALETTE.items():
    subset = df_plot[df_plot["register"] == reg]["duration_ms"]
    axes[1].hist(subset, bins=50, alpha=0.6, color=col, label=reg, edgecolor="none")
axes[1].set_title("Duration by register")
axes[1].set_xlabel("Duration (ms)")
axes[1].set_ylabel("")
axes[1].legend(title="Register", fontsize=9)

plt.tight_layout()
plt.show()

n_outliers = (df["duration_ms"] > DUR_CAP).sum()
print(f"Tokens > {DUR_CAP} ms (not shown): {n_outliers}  ({n_outliers/len(df)*100:.1f}%)")

In [ ]:
# Duration by vowel — box plot (canonical vowels only, sorted by median)
df_canon = df[df["vowel"].isin(CANONICAL_VOWELS) & (df["duration_ms"] <= DUR_CAP)].copy()
vowel_median_order = (df_canon.groupby("vowel")["duration_ms"]
                               .median()
                               .sort_values()
                               .index.tolist())

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=df_canon, x="vowel", y="duration_ms", hue="register",
            order=vowel_median_order,
            palette=REGISTER_PALETTE, linewidth=0.8,
            flierprops=dict(marker=".", markersize=2, alpha=0.3), ax=ax)
ax.set_title("Duration by vowel and register  (capped at 500 ms, sorted by IDS median)", fontsize=11)
ax.set_xlabel("Vowel  (sorted by median duration)")
ax.set_ylabel("Duration (ms)")
ax.legend(title="Register", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Duration across sessions — does IDS duration change as the infant ages?
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=df[df["duration_ms"] <= DUR_CAP],
            x="session", y="duration_ms", hue="register",
            order=SESSION_ORDER,
            palette=REGISTER_PALETTE, linewidth=0.8,
            flierprops=dict(marker=".", markersize=2, alpha=0.3), ax=ax)
ax.set_title("Duration by session and register  (capped at 500 ms)")
ax.set_xlabel("Session")
ax.set_ylabel("Duration (ms)")
ax.legend(title="Register", fontsize=9)
plt.tight_layout()
plt.show()

print("\nMedian duration (ms) by session × register:")
print(df.groupby(["session","register"], observed=True)["duration_ms"]
        .median().unstack().reindex(SESSION_ORDER).round(1))

---

## Summary and Next Steps

| Finding | Detail |
|---------|--------|
| Dataset size | 21,341 tokens, 32 speakers, 3 sessions (4m / 8m / 12m) |
| Register balance | ~82% IDS, ~18% ADS — expected given the day-long recording design |
| Vowel inventory | 14 canonical categories; 2 anomalous labels (`on-donc`, `i-copine`) to review |
| Duration | Right-skewed; median ~78 ms; long tail up to ~2.7 s; IDS tokens tend to be longer than ADS |
| Session coverage | Inspect the speaker × session matrix for any gaps before modelling |
| Acoustic columns | All `NaN` — populate by enabling `features.pitch / formants_mean` in `config/config.yaml` and re-running the pipeline |

**Immediate actions before acoustic analysis:**
1. Decide whether to keep or drop the 2 anomalous vowel labels (`on-donc`, `i-copine`).
2. Confirm speaker × session completeness is sufficient for longitudinal models.
3. Enable Praat feature extraction in `config/config.yaml` and implement the TODO blocks in `src/french_ids/praat_features.py`.